In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    GradientBoostingClassifier, StackingClassifier, VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.utils import shuffle as sk_shuffle
from lightgbm import LGBMClassifier


In [ ]:
# Loading in the train.csv
df = pd.read_csv('train.csv', na_values=["?"], low_memory=False)
df.head()

In [ ]:
# Figuring out the distribution of NaN values in the dataset
# Total NaN
total_NaN = df.isnull().sum().sum()
print(f"Total NaN values: {total_NaN:,}")
print(f"Total cells:      {df.shape[0] * df.shape[1]:,}")
print(f"Missing overall:  {total_NaN / (df.shape[0]*df.shape[1]) * 100:.1f}%\n")

# Amonut of NaN per features
nan_counts = df.isnull().sum()
nan_pct    = (nan_counts / len(df) * 100).round(2)
nan_df = pd.DataFrame({"NaN Count": nan_counts, "NaN %": nan_pct})
nan_df = nan_df[nan_df["NaN Count"] > 0].sort_values("NaN %", ascending=False)
print(nan_df.to_string())

In [ ]:
# Dropping features due to high precentage of NaN
# Additionally dropping random features that act as Identifiers (i.e. INCIDENT_DATE, FLT, REG, LUPDATE, TRANSFER)
drop_cols = [
    "BIRD_BAND_NUMBER", "ENG_4_POS", "ENROUTE_STATE", "PRECIPITATION",
    "ENG_3_POS", "LOCATION", "COMMENTS", "REMARKS",
    "INCIDENT_DATE", "FLT", "REG", "LUPDATE", "TRANSFER"
]
df.drop(columns=drop_cols, inplace=True)

In [ ]:
# Fill the numeric features's NaN value with there respective median
num_cols = ["SPEED", "HEIGHT", "DISTANCE", "AC_MASS", "NUM_ENGS",
            "ENG_1_POS", "ENG_2_POS", "EMA", "EMO"]

train_medians = {}
for col in num_cols:
    if col in df.columns:
        train_medians[col] = df[col].median()
        df[col].fillna(train_medians[col], inplace=True)


In [ ]:
# Fill the categorical features's NaN with "Unknow"
cat_cols = ["TIME_OF_DAY", "PHASE_OF_FLIGHT", "SKY", "SIZE",
            "AC_CLASS", "TYPE_ENG", "WARNED", "STATE",
            "FAAREGION", "PERSON", "SOURCE"]
for col in cat_cols:
    if col in df.columns:
        df[col].fillna("Unknown", inplace=True)

In [ ]:
# Fill NUM_STRUCK feature with 1 (most common single-bird strike)
df["NUM_STRUCK"] = pd.to_numeric(df["NUM_STRUCK"], errors="coerce")
df["NUM_STRUCK"].fillna(1, inplace=True)

# Fill NUM_SEEN feature with median
df["NUM_SEEN"] = pd.to_numeric(df["NUM_SEEN"], errors="coerce")
train_medians["NUM_SEEN"] = df["NUM_SEEN"].median()
df["NUM_SEEN"].fillna(train_medians["NUM_SEEN"], inplace=True)

In [ ]:
# Checking NaN percentage
total_NaN = df.isnull().sum().sum()
print(f"Total NaN values: {total_NaN:,}")
print(f"Total cells:      {df.shape[0] * df.shape[1]:,}")
print(f"Missing overall:  {total_NaN / (df.shape[0]*df.shape[1]) * 100:.1f}%\n")

In [ ]:
# Convert TIME (HH:MM string) to numeric HOUR, then drop original
# Missing TIME values are imputed with the median hour
df["HOUR"] = pd.to_numeric(
    df["TIME"].str.split(":").str[0], errors="coerce"
)
train_medians["HOUR"] = df["HOUR"].median()
df["HOUR"] = df["HOUR"].fillna(train_medians["HOUR"])
df.drop(columns=["TIME"], inplace=True)

# Drop LATITUDE/LONGITUDE — redundant with STATE and FAAREGION
df.drop(columns=["LATITUDE", "LONGITUDE"], inplace=True)

print("HOUR sample:", df["HOUR"].describe())
print("\nRemaining NaNs:", df.isnull().sum().sum())

In [ ]:
# Feature engineering on training set
# Phase of flight risk flags
df["IS_LANDING"]  = (df["PHASE_OF_FLIGHT"] == "Landing Roll").astype(int)
df["IS_TAKEOFF"]  = (df["PHASE_OF_FLIGHT"] == "Take-off Run").astype(int)
df["IS_APPROACH"] = (df["PHASE_OF_FLIGHT"] == "Approach").astype(int)

# Time of day risk flags
df["IS_NIGHT"] = (df["TIME_OF_DAY"] == "Night").astype(int)
df["IS_DAWN"]  = (df["TIME_OF_DAY"] == "Dawn").astype(int)
df["IS_DUSK"]  = (df["TIME_OF_DAY"] == "Dusk").astype(int)

# Domain knowledge risk features
df["IS_LARGE_BIRD"] = (df["SIZE"] == "Large").astype(int)
df["MULTI_STRIKE"]  = pd.to_numeric(df["NUM_STRUCK"], errors="coerce").gt(1).astype(int)
df["RISK_SCORE"]    = df["AC_MASS"] * df["SPEED"] * df["NUM_ENGS"]

# Interaction features
df["SPEED_X_MASS"]     = df["SPEED"] * df["AC_MASS"]
df["HEIGHT_X_SPEED"]   = df["HEIGHT"] * df["SPEED"]
df["DISTANCE_X_SPEED"] = df["DISTANCE"] * df["SPEED"]
df["MASS_X_ENGS"]      = df["AC_MASS"] * df["NUM_ENGS"]
df["HEIGHT_X_MASS"]    = df["HEIGHT"] * df["AC_MASS"]

# Binned features
df["HEIGHT_BIN"] = pd.cut(df["HEIGHT"],
    bins=[-1, 0, 500, 5000, 99999],
    labels=[0, 1, 2, 3]
).astype(int)

df["SPEED_BIN"] = pd.cut(df["SPEED"],
    bins=[-1, 0, 100, 200, 99999],
    labels=[0, 1, 2, 3]
).astype(int)

print("Feature engineering complete. Shape:", df.shape)

In [ ]:
# Class balance check
print("Class counts:")
print(df["INDICATED_DAMAGE"].value_counts())

print("\nClass percentages:")
print(df["INDICATED_DAMAGE"].value_counts(normalize=True).mul(100).round(2))


In [ ]:
# Scaling the data set
# Frequency encoding for high cardinality columns
# Replaces each category with how often it appears in training
high_card_cols = ["SPECIES", "AIRPORT", "AIRPORT_ID", "OPERATOR", "OPID", "AIRCRAFT"]
freq_encoders = {}
for col in high_card_cols:
    if col in df.columns:
        freq = df[col].value_counts() / len(df)
        freq_encoders[col] = freq
        df[col] = df[col].map(freq).fillna(0)

# Label encode remaining categorical columns
label_encoders = {}
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Separate features and target 
X = df.drop(columns=["INDICATED_DAMAGE", "INDEX_NR"])
y = df["INDICATED_DAMAGE"]

# Scale
scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Scaling complete. Shape:", X_scaled.shape)
print(X_scaled.describe().round(2))

In [ ]:
# Reading in the test dataset
test_df = pd.read_csv("test.csv", na_values=["?"], low_memory=False)
test_df.head()

In [ ]:
# Dropping same columns as training
drop_cols = [
    "BIRD_BAND_NUMBER", "ENG_4_POS", "ENROUTE_STATE", "PRECIPITATION",
    "ENG_3_POS", "LOCATION", "COMMENTS", "REMARKS",
    "INCIDENT_DATE", "FLT", "REG", "LUPDATE", "TRANSFER"
]
test_df.drop(columns=drop_cols, inplace=True)

In [ ]:
# Numeric imputation using saved training medians
num_cols = ["SPEED", "HEIGHT", "DISTANCE", "AC_MASS", "NUM_ENGS",
            "ENG_1_POS", "ENG_2_POS", "EMA", "EMO"]
for col in num_cols:
    if col in test_df.columns:
        test_df[col].fillna(train_medians[col], inplace=True)

In [ ]:
# Categorical imputation with "Unknown"
cat_fill_cols = ["TIME_OF_DAY", "PHASE_OF_FLIGHT", "SKY", "SIZE",
                 "AC_CLASS", "TYPE_ENG", "WARNED", "STATE",
                 "FAAREGION", "PERSON", "SOURCE"]
for col in cat_fill_cols:
    if col in test_df.columns:
        test_df[col].fillna("Unknown", inplace=True)

In [ ]:
# NUM_STRUCK and NUM_SEEN using saved training medians
test_df["NUM_STRUCK"] = pd.to_numeric(test_df["NUM_STRUCK"], errors="coerce")
test_df["NUM_STRUCK"].fillna(1, inplace=True)

In [ ]:
# TIME → HOUR using saved training median, drop LAT/LON
test_df["HOUR"] = pd.to_numeric(
    test_df["TIME"].str.split(":").str[0], errors="coerce"
)
test_df["HOUR"] = test_df["HOUR"].fillna(train_medians["HOUR"])
test_df.drop(columns=["TIME", "LATITUDE", "LONGITUDE"], inplace=True)

In [ ]:
# Feature engineering on test set (mirrors training)
# Phase of flight risk flags
test_df["IS_LANDING"]  = (test_df["PHASE_OF_FLIGHT"] == "Landing Roll").astype(int)
test_df["IS_TAKEOFF"]  = (test_df["PHASE_OF_FLIGHT"] == "Take-off Run").astype(int)
test_df["IS_APPROACH"] = (test_df["PHASE_OF_FLIGHT"] == "Approach").astype(int)

# Time of day risk flags
test_df["IS_NIGHT"] = (test_df["TIME_OF_DAY"] == "Night").astype(int)
test_df["IS_DAWN"]  = (test_df["TIME_OF_DAY"] == "Dawn").astype(int)
test_df["IS_DUSK"]  = (test_df["TIME_OF_DAY"] == "Dusk").astype(int)

# Domain knowledge risk features
test_df["IS_LARGE_BIRD"] = (test_df["SIZE"] == "Large").astype(int)
test_df["MULTI_STRIKE"]  = pd.to_numeric(test_df["NUM_STRUCK"], errors="coerce").gt(1).astype(int)
test_df["RISK_SCORE"]    = test_df["AC_MASS"] * test_df["SPEED"] * test_df["NUM_ENGS"]

# Interaction features
test_df["SPEED_X_MASS"]     = test_df["SPEED"] * test_df["AC_MASS"]
test_df["HEIGHT_X_SPEED"]   = test_df["HEIGHT"] * test_df["SPEED"]
test_df["DISTANCE_X_SPEED"] = test_df["DISTANCE"] * test_df["SPEED"]
test_df["MASS_X_ENGS"]      = test_df["AC_MASS"] * test_df["NUM_ENGS"]
test_df["HEIGHT_X_MASS"]    = test_df["HEIGHT"] * test_df["AC_MASS"]

# Binned features
test_df["HEIGHT_BIN"] = pd.cut(test_df["HEIGHT"],
    bins=[-1, 0, 500, 5000, 99999],
    labels=[0, 1, 2, 3]
).astype(int)

test_df["SPEED_BIN"] = pd.cut(test_df["SPEED"],
    bins=[-1, 0, 100, 200, 99999],
    labels=[0, 1, 2, 3]
).astype(int)

print("Test feature engineering complete. Shape:", test_df.shape)

In [ ]:
# Apply frequency encoding using training frequencies
for col in high_card_cols:
    if col in test_df.columns:
        test_df[col] = test_df[col].map(freq_encoders[col]).fillna(0)

# Label encode remaining categorical columns using saved encoders
for col in cat_cols:
    if col in test_df.columns:
        test_df[col] = test_df[col].astype(str).map(
            lambda x, le=label_encoders[col]: le.transform([x])[0]
            if x in le.classes_ else -1
        )

In [ ]:
# Save INDEX_NR before dropping
index_nr = test_df["INDEX_NR"]
X_test = test_df.drop(columns=["INDEX_NR"])

# Force convert every column, catches mixed type columns the dtype check misses
# THIS SECTION IS CLAUDE
for col in X_test.columns:
    try:
        X_test[col] = X_test[col].astype(float)
    except (ValueError, TypeError):
        print(f"Encoding: {col} | sample values: {X_test[col].unique()[:5]}")
        if col in label_encoders:
            X_test[col] = X_test[col].astype(str).map(
                lambda x, le=label_encoders[col]: le.transform([x])[0]
                if x in le.classes_ else -1
            )
        else:
            le = LabelEncoder()
            X_test[col] = le.fit_transform(X_test[col].astype(str))
            print(f"  Warning: {col} was not in label_encoders, fit new encoder")

# Scale
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Test set ready. Shape:", X_test_scaled.shape)
print("Remaining NaNs:", X_test_scaled.isnull().sum().sum())

In [ ]:
# Making similarity matrix to see if data is clusterable (We aren't clustering)
# Sample 5000 rows — computing on 307k rows is too slow
X_sample = X_scaled.sample(n=5000, random_state=42)

corr_matrix = X_sample.corr()

fig, ax = plt.subplots(figsize=(20, 16))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")

# Add colorbar
plt.colorbar(im, ax=ax)

# Add feature labels
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr_matrix.columns, fontsize=7)

plt.title("Feature Correlation Matrix", fontsize=16)
plt.tight_layout()
plt.show()

# Top features correlated with target
target_corr = X_scaled.corrwith(y.reset_index(drop=True)).abs().sort_values(ascending=False)
print("Top 15 features most correlated with INDICATED_DAMAGE:")
print(target_corr.head(15))


In [ ]:
# Dropping featues that have near-zero variance
zero_var_cols = ["NUM_SEEN", "NUM_STRUCK", "OUT_OF_RANGE_SPECIES"]
for col in zero_var_cols:
    if col in X_scaled.columns:
        X_scaled = X_scaled.drop(columns=[col])

# Drop zero-variance columns from test set
for col in zero_var_cols:
    if col in X_test_scaled.columns:
        X_test_scaled = X_test_scaled.drop(columns=[col])

print("Remaining training features:", X_scaled.shape[1])
print("Remaining test features:", X_test_scaled.shape[1])

In [ ]:
# Fill any remaining NaNs with 0 (= median after RobustScaler)
X_scaled = X_scaled.fillna(0)
X_test_scaled = X_test_scaled.fillna(0)

print("NaNs remaining in X_scaled:", X_scaled.isnull().sum().sum())
print("NaNs remaining in X_test_scaled:", X_test_scaled.isnull().sum().sum())

# Reset and clean y before model training
y = df["INDICATED_DAMAGE"].reset_index(drop=True)

# Drop any rows where target is NaN
mask = y.notna()
y = y[mask].reset_index(drop=True)
X_scaled = X_scaled[mask].reset_index(drop=True)

print("NaNs in y:", y.isnull().sum())
print("X_scaled shape:", X_scaled.shape)
print("Class counts:\n", y.value_counts())

In [ ]:
damage_idx    = y[y == 1].index
no_damage_idx = y[y == 0].index

n_damage    = len(damage_idx)
n_no_damage = int(n_damage * 1.5)  # 60% no-damage, 40% damage

no_damage_sampled_idx = no_damage_idx.to_series().sample(
    n=n_no_damage, random_state=42
).index

keep_idx = damage_idx.tolist() + no_damage_sampled_idx.tolist()
X_scaled = X_scaled.loc[keep_idx].reset_index(drop=True)
y        = y.loc[keep_idx].reset_index(drop=True)

X_scaled, y = sk_shuffle(X_scaled, y, random_state=42)
X_scaled = X_scaled.reset_index(drop=True)
y        = y.reset_index(drop=True)

print("After undersampling:")
print("  Total rows:    ", len(y))
print("  Damage (1):    ", (y == 1).sum(), f"({(y==1).mean()*100:.1f}%)")
print("  No damage (0): ", (y == 0).sum(), f"({(y==0).mean()*100:.1f}%)")

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv  = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ── Hyperparameter grid ───────────────────────────────────────────
lgbm_params = {
    "n_estimators":      [500, 700, 1000],
    "learning_rate":     [0.01, 0.05, 0.1],
    "max_depth":         [5, 7, 10],
    "num_leaves":        [31, 63, 127],
    "subsample":         [0.7, 0.8, 1.0],
    "colsample_bytree":  [0.7, 0.8, 1.0],
    "reg_alpha":         [0, 0.1, 1.0],
    "reg_lambda":        [0, 0.1, 1.0],
    "min_child_samples": [10, 20, 50]
}

def undersample(X, y, ratio=1.5, seed=42):
    """Undersample majority class to get 60/40 split."""
    damage_idx    = y[y == 1].index
    no_damage_idx = y[y == 0].index
    n_no_damage   = int(len(damage_idx) * ratio)
    sampled_idx   = no_damage_idx.to_series().sample(n=n_no_damage, random_state=seed).index
    keep_idx      = damage_idx.tolist() + sampled_idx.tolist()
    X_u = X.loc[keep_idx].reset_index(drop=True)
    y_u = y.loc[keep_idx].reset_index(drop=True)
    return sk_shuffle(X_u, y_u, random_state=seed)

# ── Nested CV — undersample inside each fold ─────────────────────
# Validation always uses original distribution — no undersampling on val
lgbm_outer_scores = []

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_scaled, y)):
    print(f"\nOuter Fold {fold+1}/5...")

    X_tr_full = X_scaled.iloc[train_idx].reset_index(drop=True)
    y_tr_full = y.iloc[train_idx].reset_index(drop=True)
    X_val     = X_scaled.iloc[val_idx]
    y_val     = y.iloc[val_idx]

    # Undersample training fold only
    X_tr_under, y_tr_under = undersample(X_tr_full, y_tr_full)

    lgbm_search = RandomizedSearchCV(
        LGBMClassifier(class_weight="balanced", random_state=42,
                       device="gpu", n_jobs=-1, verbose=-1),
        lgbm_params, n_iter=20, scoring="balanced_accuracy",
        cv=inner_cv, n_jobs=-1, random_state=42
    )
    lgbm_search.fit(X_tr_under, y_tr_under)
    lgbm_score = balanced_accuracy_score(
        y_val, lgbm_search.best_estimator_.predict(X_val)
    )
    lgbm_outer_scores.append(lgbm_score)
    print(f"  Fold score: {lgbm_score:.4f} | best params: {lgbm_search.best_params_}")

print(f"\n══════════════════════════════════════════")
print(f"  LightGBM Nested CV Balanced Accuracy")
print(f"  {np.mean(lgbm_outer_scores):.4f} +/- {np.std(lgbm_outer_scores):.4f}")
print(f"══════════════════════════════════════════")

# ── Final model on full undersampled dataset ──────────────────────
print("\nFitting final LightGBM on full undersampled dataset...")
X_final, y_final = undersample(X_scaled, y)
print(f"Final training size: {X_final.shape}")
print(f"Class split: {y_final.value_counts().to_dict()}")

lgbm_final = RandomizedSearchCV(
    LGBMClassifier(class_weight="balanced", random_state=42,
                   device="gpu", n_jobs=-1, verbose=-1),
    lgbm_params, n_iter=20, scoring="balanced_accuracy",
    cv=inner_cv, n_jobs=-1, random_state=42
)
lgbm_final.fit(X_final, y_final)
best_lgbm = lgbm_final.best_estimator_
print(f"\nBest params: {lgbm_final.best_params_}")

# ── Feature importance — drop bottom 10 features ─────────────────
feat_importance = pd.Series(
    best_lgbm.feature_importances_,
    index=X_final.columns
).sort_values(ascending=False)

print("\nTop 15 features:")
print(feat_importance.head(15))

bottom_features = feat_importance.tail(10).index.tolist()
print("\nDropping bottom 10 features:", bottom_features)

X_scaled      = X_scaled.drop(columns=bottom_features)
X_test_scaled = X_test_scaled.drop(columns=bottom_features)
X_final       = X_final.drop(columns=bottom_features)
print("Shape after dropping:", X_scaled.shape)

# Retrain on cleaned features
print("\nRetraining on cleaned feature set...")
lgbm_final.fit(X_final, y_final)
best_lgbm = lgbm_final.best_estimator_
print("Done.")

In [ ]:
# ── Threshold tuning on LightGBM ─────────────────────────────────
y_proba_train = best_lgbm.predict_proba(X_final)[:, 1]
y_proba_test  = best_lgbm.predict_proba(X_test_scaled)[:, 1]

best_thresh, best_score = 0.5, 0
for thresh in [i/1000 for i in range(100, 600)]:
    preds = (y_proba_train >= thresh).astype(int)
    score = balanced_accuracy_score(y_final, preds)
    if score > best_score:
        best_score = score
        best_thresh = thresh

print("══════════════════════════════════════════")
print("         TRAIN BALANCED ACCURACY          ")
print("══════════════════════════════════════════")
print(f"  LightGBM:       {best_score:.4f}")
print(f"  Best threshold: {best_thresh:.3f}")
print("══════════════════════════════════════════")

In [ ]:
# ── Generate submission ──────────────────────────────────────────
y_pred_final = (y_proba_test >= best_thresh).astype(int)

submission = pd.DataFrame({
    "INDEX_NR": index_nr.values,
    "INDICATED_DAMAGE": y_pred_final
})
submission.to_csv("submission_lgbm.csv", index=False)

print("Submission saved!")
print("\nPrediction distribution:")
print(submission["INDICATED_DAMAGE"].value_counts())
print(submission["INDICATED_DAMAGE"].value_counts(normalize=True).mul(100).round(2))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# DECISION STUMP (Decision Tree with max_depth=1)
stump_params = {
    "class_weight": ["balanced"],
    "criterion":    ["gini", "entropy"],
    "splitter":     ["best", "random"]
}
stump_search = RandomizedSearchCV(
    DecisionTreeClassifier(max_depth=1, random_state=42),
    stump_params, n_iter=6, scoring="balanced_accuracy",
    cv=cv, n_jobs=-1, random_state=42
)
stump_search.fit(X_scaled, y)
best_stump = stump_search.best_estimator_

stump_scores = cross_val_score(best_stump, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Decision Stump ──")
print(f"  Best params: {stump_search.best_params_}")
print(f"  CV Balanced Accuracy: {stump_scores.mean():.4f} +/- {stump_scores.std():.4f}")

# BAGGING
bagging_params = {
    "n_estimators":      [50, 100, 200],
    "max_samples":       [0.7, 0.8, 1.0],
    "max_features":      [0.7, 0.8, 1.0],
    "bootstrap":         [True, False]
}
bagging_search = RandomizedSearchCV(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42),
        random_state=42
    ),
    bagging_params, n_iter=10, scoring="balanced_accuracy",
    cv=cv, n_jobs=-1, random_state=42
)
bagging_search.fit(X_scaled, y)
best_bagging = bagging_search.best_estimator_

bagging_scores = cross_val_score(best_bagging, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Bagging ──")
print(f"  Best params: {bagging_search.best_params_}")
print(f"  CV Balanced Accuracy: {bagging_scores.mean():.4f} +/- {bagging_scores.std():.4f}")

# RANDOM FOREST
rf_params = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"]
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    rf_params, n_iter=15, scoring="balanced_accuracy",
    cv=cv, n_jobs=-1, random_state=42
)
rf_search.fit(X_scaled, y)
best_rf = rf_search.best_estimator_

rf_scores = cross_val_score(best_rf, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Random Forest ──")
print(f"  Best params: {rf_search.best_params_}")
print(f"  CV Balanced Accuracy: {rf_scores.mean():.4f} +/- {rf_scores.std():.4f}")

# BOOSTING (Gradient Boosting)
boost_params = {
    "n_estimators":      [100, 200, 300],
    "learning_rate":     [0.01, 0.05, 0.1, 0.2],
    "max_depth":         [3, 4, 5],
    "subsample":         [0.7, 0.8, 1.0],
    "min_samples_leaf":  [1, 2, 4]
}
boost_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    boost_params, n_iter=15, scoring="balanced_accuracy",
    cv=cv, n_jobs=-1, random_state=42
)
boost_search.fit(X_scaled, y)
best_boost = boost_search.best_estimator_

boost_scores = cross_val_score(best_boost, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Boosting ──")
print(f"  Best params: {boost_search.best_params_}")
print(f"  CV Balanced Accuracy: {boost_scores.mean():.4f} +/- {boost_scores.std():.4f}")

# MULTICLASS PARTITIONING (One-vs-Rest)
ovr_params = {
    "estimator__C":        [0.01, 0.1, 1, 10],
    "estimator__solver":   ["lbfgs", "saga"],
    "estimator__max_iter": [200, 500]
}
ovr_search = RandomizedSearchCV(
    OneVsRestClassifier(LogisticRegression(class_weight="balanced", random_state=42)),
    ovr_params, n_iter=10, scoring="balanced_accuracy",
    cv=cv, n_jobs=-1, random_state=42
)
ovr_search.fit(X_scaled, y)
best_ovr = ovr_search.best_estimator_

ovr_scores = cross_val_score(best_ovr, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Multiclass Partitioning (OvR) ──")
print(f"  Best params: {ovr_search.best_params_}")
print(f"  CV Balanced Accuracy: {ovr_scores.mean():.4f} +/- {ovr_scores.std():.4f}")

# STACKING
stacking = StackingClassifier(
    estimators=[
        ("stump",   best_stump),
        ("bagging", best_bagging),
        ("rf",      best_rf),
        ("boost",   best_boost),
        ("ovr",     best_ovr)
    ],
    final_estimator=LogisticRegression(class_weight="balanced", random_state=42),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_scaled, y)

stacking_scores = cross_val_score(stacking, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Stacking ──")
print(f"  CV Balanced Accuracy: {stacking_scores.mean():.4f} +/- {stacking_scores.std():.4f}")


# VOTING ENSEMBLE (best models combined)
voting = VotingClassifier(
    estimators=[
        ("stump",    best_stump),
        ("bagging",  best_bagging),
        ("rf",       best_rf),
        ("boost",    best_boost),
        ("ovr",      best_ovr),
        ("stacking", stacking)
    ],
    voting="soft",
    n_jobs=-1
)
voting.fit(X_scaled, y)

voting_scores = cross_val_score(voting, X_scaled, y, cv=cv, scoring="balanced_accuracy")
print("── Voting Ensemble ──")
print(f"  CV Balanced Accuracy: {voting_scores.mean():.4f} +/- {voting_scores.std():.4f}/")

results = {
    "Decision Stump":          stump_scores.mean(),
    "Bagging":                 bagging_scores.mean(),
    "Random Forest":           rf_scores.mean(),
    "Boosting":                boost_scores.mean(),
    "Multiclass Part. (OvR)": ovr_scores.mean(),
    "Stacking":                stacking_scores.mean(),
    "Voting Ensemble":         voting_scores.mean()
}
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<28} {score:.4f}")

print(f"Best model: {max(results, key=results.get)}")
print(f"Best Balanced Accuracy: {max(results.values()):.4f}")

#  Generate predictions using best voting ensemble
y_pred = voting.predict(X_test_scaled)

# Build submission file
submission = pd.DataFrame({
    "INDEX_NR": index_nr.values,
    "INDICATED_DAMAGE": y_pred
})

submission.to_csv("submission.csv", index=False)
print(submission["INDICATED_DAMAGE"].value_counts())
print(submission["INDICATED_DAMAGE"].value_counts(normalize=True).mul(100).round(2))